In [0]:
%sql
MERGE INTO bootcamp_matias.gold.dim_zona AS t
USING(
SELECT
DISTINCT
  zona,
  ciudad,
  'Buenos Aires' AS provincia,
  CURRENT_DATE() AS fecha_inicio,
  '9999-12-31' AS fecha_final,
  TRUE AS es_actual
FROM bootcamp_matias.silver.propiedades
) AS s

ON t.ciudad = s.ciudad

WHEN MATCHED AND t.fecha_inicio != s.fecha_inicio THEN
UPDATE SET
t.fecha_final=CURRENT_DATE(),
t.es_actual=FALSE

WHEN NOT MATCHED THEN
INSERT
(
  zona,
  ciudad,
  provincia,
  fecha_inicio,
  fecha_final,
  es_actual,
  _created_at
)
values (
s.zona,
s.ciudad,
s.provincia,
CURRENT_DATE(),
'9999-12-31', 
TRUE,
CURRENT_TIMESTAMP()
)

In [0]:
%sql
-- Insertar el nuevo registro solo para zonas que fueron cerradas en esta ejecución
INSERT INTO bootcamp_matias.gold.dim_zona (zona,ciudad,provincia,fecha_inicio,fecha_final, es_actual)
SELECT 
  s.zona,
  s.ciudad,
  'Buenos Aires' as provincia,
  CURRENT_DATE() AS fecha_inicio,
  '9999-12-31' AS fecha_final,
  TRUE AS es_actual
FROM bootcamp_matias.silver.propiedades s
INNER JOIN bootcamp_matias.gold.dim_zona t
  ON s.ciudad = t.ciudad
  AND t.es_actual=FALSE
  AND t.fecha_final=CURRENT_DATE()
WHERE NOT EXISTS (
  SELECT 1
  FROM bootcamp_matias.gold.dim_zona t2
  WHERE t2.ciudad = s.ciudad
    AND t2.es_actual=TRUE
    AND t2.fecha_inicio = t2.fecha_inicio
)